# CS-781 Phase 1 — Architecture 1: Encoder/Prefix-LM (~120M params, 500M-token plan)

This is the scaled version of the tiny sanity pipeline.

**What changes from the sanity notebook**
- uses **all four official Phase-1 training sources**;
- targets about **500M BPE training tokens** by default;
- trains a **~120M parameter encoder-prefix language model**;
- uses a larger custom byte-level BPE vocabulary;
- streams JSON with `ijson` and writes `uint16` binary token files instead of keeping hundreds of millions of Python integers in RAM;
- uses PyTorch scaled-dot-product attention;
- projects to the vocabulary only on suffix positions used by the loss;
- uses FP16, fused AdamW when available, resumable checkpoints, and a fixed shuffled block order;
- generation uses a real **KV cache**, so 128-word continuation generation is much faster than the tiny notebook.

The language-model objective is still the same idea:
- prompt/prefix tokens attend bidirectionally to the prefix;
- continuation tokens attend to the prefix and causally to previous continuation tokens;
- loss is applied only to continuation next-token targets.

The default source budget is:

- Supreme Court: 180M tokens
- High Courts: 180M tokens
- Tribunals: 90M tokens
- Acts: 50M tokens
- **Total: 500M tokens**

Run the notebook cell-by-cell. The expensive stages automatically reuse existing tokenizer/data/checkpoints if they already exist.

**Revision before full GPU run:** context length and prefix lengths are now measured from the official 512-word prompts; dropout is 0; inference stays within the trained positional range; checkpoint writes are atomic; optional private-HF checkpoint backup is supported.

---
### Revision 3 — fixes applied on top of the REVISED notebook (everything else is unchanged)
1. **Generation is batched** (left-padded prompts, per-sample positions + key-padding mask, KV cache) and **never aborts**:
   a short output is counted and reported, stragglers may use positions up to `MAX_POSITIONS`, the official run writes a
   resumable `prediction_partial.json`.
2. **HF checkpoint backup is safe**: repo auto-created, errors never kill training, history squashed every few uploads,
   uploads every 2,000 steps, and a fresh session **restores `latest.pt` automatically** from the repo.
3. **MLM share in the objective** (`P_MLM`, default 0.30; reserved id `<s>` as mask token — no tokenizer change) so the
   prefix/encoder tokens get a direct loss for Phases 2-5 and 7. `P_MLM = 0.0` gives the pure prefix-LM.
4. **Throughput**: micro-batch 4 × accumulation 4 (same 16 sequences/update). **WSD** LR schedule (cosine still available).
5. **Pilot switch** `TRAIN_BLOCK_LIMIT` + `RUN_NAME`, so a short pilot never mixes with the full run.
6. **Validation**: per-source prefix-LM loss + MLM loss logged; 40 local generation pairs per source; per-source BERTScore/ROUGE-L;
   `train_log.jsonl`, `run_config.json`, `eval_log.jsonl` written for the report.
7. Smaller items: no weight decay on biases/LayerNorm; sources are no longer downloaded twice; HF cache on `/kaggle/temp`.


In [1]:
# Cell 1 — Install dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ijson",
    "huggingface_hub",
    "transformers==4.48.3",
    "bert-score==0.3.13",
])

print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.9 MB/s eta 0:00:00
Dependencies ready.


In [2]:
# Cell 2 — Imports + full-run configuration
import os
# Keep the multi-GB HF download cache on Kaggle's large temp disk when it exists.
# (Must be set BEFORE huggingface_hub is imported.)
if os.path.isdir("/kaggle/temp"):
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
import re, gc, json, math, time, random, zipfile, heapq, shutil
from pathlib import Path
from collections import Counter

import ijson
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, login

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Kaggle secret should be named HF_TOKEN.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except Exception as e:
    print("HF secret unavailable; public access will be attempted.", repr(e))

if HF_TOKEN:
    login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "sc":   "Phase_1_Pretraining/Train/train_supreme_court.json",
    "hc":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tr":   "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}

# ----------------------------- output paths
ROOT = Path("/kaggle/working/cs781_encoder_120m")
DATA_DIR = ROOT / "data"
# Use a different RUN_NAME for a pilot (e.g. "enc_pilot") so pilot and full-run checkpoints never mix.
RUN_NAME = "enc_full"
CKPT_DIR = ROOT / "checkpoints" / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = DATA_DIR / "tokenizer.json"
TRAIN_BIN = DATA_DIR / "train_all.bin"
VAL_BIN = DATA_DIR / "val_all.bin"
META_PATH = DATA_DIR / "meta.json"
VAL_GEN_PATH = DATA_DIR / "val_gen.json"
VAL_REF_PATH = DATA_DIR / "val_ref.json"

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
FINAL_CKPT = CKPT_DIR / "final.pt"

# Optional remote checkpoint persistence.
# Leave None unless you create a PRIVATE Hugging Face model repo you own,
# e.g. "your-username/cs781-encoder-a1-checkpoints".
HF_CHECKPOINT_REPO = None
HF_UPLOAD_EVERY = 2000                # each upload adds ~1.4 GB to the repo history ...
HF_SQUASH_EVERY_N_UPLOADS = 4         # ... so squash the history every N uploads (0 = never) to bound storage

LOCAL_PRED = ROOT / "val_pred.json"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# ----------------------------- tokenizer
VOCAB_SIZE = 24_576
TOKENIZER_CHAR_BUDGET = {
    "sc":   16_000_000,
    "hc":   16_000_000,
    "tr":   10_000_000,
    "acts":  6_000_000,
}
TOKENIZER_MIN_COUNT = 2

# ----------------------------- pretraining token budget
TRAIN_TOKEN_BUDGET = {
    "sc":   180_000_000,
    "hc":   180_000_000,
    "tr":    90_000_000,
    "acts":  50_000_000,
}
TARGET_TOTAL_TOKENS = sum(TRAIN_TOKEN_BUDGET.values())  # 500M
VAL_TOKENS_PER_SOURCE = 500_000
VAL_GEN_PER_SOURCE = 40   # prompt/reference pairs per source for local BERTScore (4 sources -> up to 160)
VAL_PERMILLE = 5

# JSON/source files can be large. True saves disk but may require re-downloads.
DELETE_SOURCE_AFTER_STAGE = True

# CPU encoding parallelism; 4 is conservative for Kaggle RAM.
ENCODE_PROCESSES = min(4, os.cpu_count() or 2)

# ----------------------------- model
# IMPORTANT: SEQ_LEN and the prefix-token range are measured from the
# official 512-word prompts in Cell 6. Do not hard-code them before that.
SEQ_LEN = None
MAX_POSITIONS = 2048

D_MODEL = 768
N_LAYERS = 14
N_HEADS = 12
FFN_DIM = 3072
DROPOUT = 0.0

# Filled by Cell 6 from the actual tokenizer statistics.
PROMPT_P10_TOKENS = None
PROMPT_P95_TOKENS = None
EXPECTED_CONT_TOKENS = None
PREFIX_TOKENS_MIN = None
PREFIX_TOKENS_MAX = None
PROMPT_KEEP_TOKENS = None

# ----------------------------- optimization
MICRO_BATCH_SIZE = 4       # 4 sequences per forward (was 1: under-used the GPU). If you hit CUDA OOM use 2 (and ACCUM 8) or 1 (and 16).
GRAD_ACCUM_STEPS = 4       # effective batch = MICRO_BATCH_SIZE x GRAD_ACCUM_STEPS = 16 sequences/update (16 x SEQ_LEN tokens)
LEARNING_RATE = 3e-4
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0

# Learning-rate schedule: "wsd" = warmup -> constant -> linear decay over the last WSD_DECAY_FRAC of the run. Unlike
# cosine it lets you stop early or extend a run without ending at a high learning rate. "cosine" = previous behaviour.
LR_SCHEDULE = "wsd"
WSD_DECAY_FRAC = 0.20

# Objective mix. With probability P_MLM a micro-batch is trained as MLM (fully bidirectional attention; MLM_PROB of the
# tokens are replaced by a reserved mask id) so the prefix/encoder representations get a DIRECT loss (needed for the
# later NER / RR / CJPE / LSI / PCR phases). Otherwise it is the prefix-LM objective used so far.  P_MLM = 0.0 -> pure prefix-LM.
P_MLM = 0.30
MLM_PROB = 0.20

# Pilot switch: use only the first N blocks of the shuffled order, e.g. TRAIN_BLOCK_LIMIT = 30_000_000 // SEQ_LEN
# (~30M tokens). The LR schedule is defined over the limited run, and WSD lets you extend it later. Use another RUN_NAME.
TRAIN_BLOCK_LIMIT = None

LOG_EVERY = 50
VAL_EVERY = 1000
SAVE_EVERY = 500
VAL_EVAL_BLOCKS = 256

# ----------------------------- generation
TARGET_GENERATION_WORDS = 128
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
GEN_TOP_K = 0
GEN_REP_PENALTY = 1.03
GEN_MAX_NEW_TOKENS = 768
GEN_BATCH_SIZE = 16          # prompts generated together (left-padded, KV cache)
OFFICIAL_PARTIAL_EVERY = 4   # official-test generation: write a resumable partial file every N batches

# Evaluation / packaging
FAST_EVAL_NO_BERT = False
BERTSCORE_BATCH_SIZE = 8
RUN_OFFICIAL_TEST = False

print("Output:", ROOT)
print("Target train tokens:", f"{TARGET_TOTAL_TOKENS/1e6:.0f}M")

Device: cuda
GPU: Tesla T4
HF token loaded.
Output: /kaggle/working/cs781_encoder_120m
Target train tokens: 500M


In [3]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

Tokenizer class ready.


In [4]:
# Cell 4 — Streaming source utilities + robust test discovery
def iter_docs(path):
    """Yield document text from the official top-level JSON array."""
    with open(path, "rb") as f:
        for obj in ijson.items(f, "item"):
            text = obj.get("text", "")
            if isinstance(text, str) and text.strip():
                yield text

def fetch_source(key):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=SOURCE_FILES[key],
        repo_type="dataset",
        token=HF_TOKEN,
    )

def find_test_file():
    files = HfApi().list_repo_files(
        HF_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    candidates = [
        f for f in files
        if f.startswith("Phase_1_Pretraining/Test") and f.endswith(".json")
    ]
    if not candidates:
        raise RuntimeError("Could not discover Phase-1 test JSON.")
    chosen = candidates[0]
    print("Official test file:", chosen)
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=chosen,
        repo_type="dataset",
        token=HF_TOKEN,
    )

def maybe_delete_download(path):
    if not DELETE_SOURCE_AFTER_STAGE:
        return
    try:
        real = os.path.realpath(path)
        if os.path.exists(real):
            os.remove(real)
            print("Deleted cached source to save disk:", real)
    except Exception as e:
        print("Could not delete source cache:", repr(e))

def source_census(key, sample_docs=1000):
    path = fetch_source(key)
    size = os.path.getsize(path)
    n = chars = words = 0
    lens = []
    for doc in iter_docs(path):
        n += 1
        chars += len(doc)
        w = len(doc.split())
        words += w
        lens.append(w)
        if n >= sample_docs:
            break
    avg_chars = chars / max(n,1)
    # JSON overhead correction; only used to choose approximate uniform sampling probability.
    est_docs = int(size / max(1.0, avg_chars * 1.03))
    est_chars = avg_chars * est_docs
    return {
        "file_bytes":size,
        "sample_docs":n,
        "avg_chars":avg_chars,
        "avg_words":words/max(n,1),
        "est_docs":est_docs,
        "est_chars":est_chars,
        "p10_p50_p90_words":[int(np.percentile(lens,q)) for q in (10,50,90)] if lens else [],
    }

print("Streaming utilities ready.")

Streaming utilities ready.


In [5]:
# Cell 5 — Census + train BPE on a uniform sample from ALL FOUR sources
CENSUS_PATH = DATA_DIR / "census.json"

if CENSUS_PATH.exists():
    census = json.load(open(CENSUS_PATH))
    print("Loaded existing census.")
else:
    census = {}
    for key in SOURCE_FILES:
        t0=time.time()
        census[key]=source_census(key, sample_docs=1000)
        print(
            key,
            f"{census[key]['file_bytes']/1e9:.2f} GB",
            "avg_chars", f"{census[key]['avg_chars']:,.0f}",
            "est_docs", f"{census[key]['est_docs']:,}",
            "words p10/50/90", census[key]["p10_p50_p90_words"],
            f"({time.time()-t0:.0f}s)"
        )
        # Keep downloaded file for tokenizer stage in this same session.
    json.dump(census, open(CENSUS_PATH,"w"), indent=2)

if TOKENIZER_PATH.exists():
    tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
    print("Loaded existing tokenizer:", len(tokenizer))
else:
    rng = random.Random(SEED)

    def tokenizer_text_stream():
        for key, char_budget in TOKENIZER_CHAR_BUDGET.items():
            path = fetch_source(key)
            est_chars = max(1.0, census[key]["est_chars"])
            # Slight oversampling makes it unlikely we finish far below target.
            q = min(1.0, 1.05 * char_budget / est_chars)
            got = 0
            seen = 0
            print(f"[{key}] tokenizer sample probability={q:.6f}")
            for doc in iter_docs(path):
                seen += 1
                if rng.random() < q:
                    # Cap unusually huge documents so one case cannot dominate merge learning.
                    d = doc[:30000]
                    got += len(d)
                    yield d
                    if got >= char_budget:
                        break
            print(f"[{key}] tokenizer chars={got/1e6:.1f}M | docs scanned={seen:,}", flush=True)
            # Do NOT delete here: Cell 7 re-reads this file (all four are already on disk after the census),
            # so keeping it avoids downloading every source twice. Cell 7 deletes each file when finished.

    tokenizer = LegalBPE(VOCAB_SIZE)
    t0=time.time()
    tokenizer.fit(tokenizer_text_stream(), min_count=TOKENIZER_MIN_COUNT)
    tokenizer.save_json(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")
    print(f"Tokenizer training elapsed: {(time.time()-t0)/60:.1f} min")

# Exact byte-level round-trip sanity.
probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
roundtrip = tokenizer.decode(tokenizer.encode(probe))
print("Round-trip exact:", roundtrip == probe)
print("Vocab:", len(tokenizer))

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

sc 1.32 GB avg_chars 36,184 est_docs 35,486 words p10/50/90 [1580, 3869, 11757] (9s)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

hc 10.67 GB avg_chars 2,648 est_docs 3,913,049 words p10/50/90 [68, 206, 960] (42s)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tr 6.48 GB avg_chars 27,783 est_docs 226,580 words p10/50/90 [238, 3432, 9831] (29s)


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts 0.73 GB avg_chars 44,418 est_docs 15,929 words p10/50/90 [407, 2827, 14848] (13s)
[sc] tokenizer sample probability=0.013084
  BPE corpus read: 10M chars | unique units=78,714
[sc] tokenizer chars=11.1M | docs scanned=55,528
[hc] tokenizer sample probability=0.001621
  BPE corpus read: 20M chars | unique units=146,851
[hc] tokenizer chars=13.8M | docs scanned=2,536,738
[tr] tokenizer sample probability=0.001668
  BPE corpus read: 30M chars | unique units=216,937
[tr] tokenizer chars=9.0M | docs scanned=869,132
[acts] tokenizer sample probability=0.008904
[acts] tokenizer chars=2.9M | docs scanned=18,822
BPE training units after min_count=2: 103,942
Counting time: 1.4 min
merge 1/24,316 | freq=760,292 | vocab=261
merge 1,000/24,316 | freq=2,684 | vocab=1,260
merge 2,000/24,316 | freq=1,112 | vocab=2,260
merge 3,000/24,316 | freq=635 | vocab=3,260
merge 4,000/24,316 | freq=415 | vocab=4,260
merge 5,000/24,316 | freq=297 | vocab=5,260
merge 6,000/24,316 | freq=225 | vocab=6,260
merge

In [6]:
# Cell 6 — Measure OFFICIAL 512-word prompts and choose the REAL training context
test_path = find_test_file()
test_data_for_stats = json.load(open(test_path, encoding="utf-8"))

prompt_token_lens = []
prompt_word_lens = []

for rec in test_data_for_stats.values():
    text = rec["text"]
    prompt_token_lens.append(len(tokenizer.encode(text)))
    prompt_word_lens.append(len(text.split()))

tpw = sum(prompt_token_lens) / max(1, sum(prompt_word_lens))

PROMPT_P10_TOKENS = int(np.percentile(prompt_token_lens, 10))
PROMPT_P95_TOKENS = int(np.percentile(prompt_token_lens, 95))

# Budget for a 128-WORD continuation, with 35% headroom for fragmentation.
raw_cont_budget = int(math.ceil(128 * tpw * 1.35))
EXPECTED_CONT_TOKENS = int(math.ceil(raw_cont_budget / 64) * 64)

# Train on a context that actually covers a p95 512-word prompt + expected continuation.
SEQ_LEN = int(math.ceil((PROMPT_P95_TOKENS + EXPECTED_CONT_TOKENS) / 128) * 128)

if SEQ_LEN > MAX_POSITIONS:
    raise RuntimeError(
        f"Measured SEQ_LEN={SEQ_LEN} exceeds MAX_POSITIONS={MAX_POSITIONS}. "
        "Increase MAX_POSITIONS before preprocessing/training."
    )

# Absolute prefix lengths, not arbitrary percentages.
# This exposes training to approximately the same BPE prompt-length range as the test.
PREFIX_TOKENS_MIN = max(64, min(PROMPT_P10_TOKENS, SEQ_LEN - 64))
PREFIX_TOKENS_MAX = max(
    PREFIX_TOKENS_MIN,
    min(PROMPT_P95_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)

# Inference prompt is kept inside the same positional range used in training.
PROMPT_KEEP_TOKENS = min(PROMPT_P95_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)

print("Official prompts:", len(prompt_token_lens))
print(
    "BPE tokens min/mean/p10/p95/max:",
    min(prompt_token_lens),
    f"{np.mean(prompt_token_lens):.0f}",
    PROMPT_P10_TOKENS,
    PROMPT_P95_TOKENS,
    max(prompt_token_lens),
)
print("Tokens/word:", round(tpw, 3))
print("Expected 128-word continuation budget:", EXPECTED_CONT_TOKENS, "BPE tokens")
print("CHOSEN SEQ_LEN:", SEQ_LEN)
print("Training prefix range:", PREFIX_TOKENS_MIN, "to", PREFIX_TOKENS_MAX, "BPE tokens")
print("Inference prompt keep:", PROMPT_KEEP_TOKENS, "BPE tokens")
print("MAX_POSITIONS:", MAX_POSITIONS)

assert PREFIX_TOKENS_MAX + EXPECTED_CONT_TOKENS <= SEQ_LEN
assert PROMPT_KEEP_TOKENS + EXPECTED_CONT_TOKENS <= SEQ_LEN

Official test file: Phase_1_Pretraining/Test/test.json


test.json: 0.00B [00:00, ?B/s]

Official prompts: 1000
BPE tokens min/mean/p10/p95/max: 1030 1082 1050 1139 1367
Tokens/word: 2.114
Expected 128-word continuation budget: 384 BPE tokens
CHOSEN SEQ_LEN: 1536
Training prefix range: 1050 to 1139 BPE tokens
Inference prompt keep: 1139 BPE tokens
MAX_POSITIONS: 2048


In [7]:
# Cell 7 — Stream/tokenize ALL FOUR sources to uint16 train/val binaries
# This is the CPU-heavy preprocessing stage. It is resumable at the FILE level:
# if meta.json + both bins already exist, this entire stage is skipped.

_WORKER_TOKENIZER = None
_WORKER_EOS = None

def _encode_worker(text):
    ids = _WORKER_TOKENIZER.encode(text)
    ids.append(_WORKER_EOS)
    return np.asarray(ids, dtype=np.uint16)

def make_pool():
    global _WORKER_TOKENIZER, _WORKER_EOS
    _WORKER_TOKENIZER = tokenizer
    _WORKER_EOS = tokenizer.eos_token_id

    if ENCODE_PROCESSES <= 1:
        return None

    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
        return ctx.Pool(ENCODE_PROCESSES)
    except Exception as e:
        print("Multiprocessing unavailable; falling back to single process:", repr(e))
        return None

def encode_many(texts, pool):
    if pool is None:
        return [_encode_worker(x) for x in texts]
    return list(pool.imap(_encode_worker, texts, chunksize=8))

if META_PATH.exists() and TRAIN_BIN.exists() and VAL_BIN.exists() and VAL_GEN_PATH.exists() and VAL_REF_PATH.exists():
    meta = json.load(open(META_PATH))
    print("Existing tokenized corpus found; skipping preprocessing.")
    print(json.dumps(meta, indent=2)[:2500])
else:
    # Start fresh so partial old bins cannot be mistaken for complete data.
    for p in [TRAIN_BIN, VAL_BIN, VAL_GEN_PATH, VAL_REF_PATH, META_PATH]:
        if p.exists():
            p.unlink()

    f_train = open(TRAIN_BIN, "wb")
    f_val = open(VAL_BIN, "wb")
    pool = make_pool()

    meta = {
        "vocab_size":len(tokenizer),
        "seq_len":SEQ_LEN,
        "prompt_p10_tokens":PROMPT_P10_TOKENS,
        "prompt_p95_tokens":PROMPT_P95_TOKENS,
        "expected_cont_tokens":EXPECTED_CONT_TOKENS,
        "prefix_tokens_min":PREFIX_TOKENS_MIN,
        "prefix_tokens_max":PREFIX_TOKENS_MAX,
        "sources":{},
        "target_total_tokens":TARGET_TOTAL_TOKENS,
    }
    val_gen = {}
    val_ref = {}

    rng = random.Random(SEED + 99)

    try:
        for key, target in TRAIN_TOKEN_BUDGET.items():
            path = fetch_source(key)

            # Estimate tokens/char from the first ~200 documents.
            probe_docs = []
            probe_chars = 0
            for doc in iter_docs(path):
                probe_docs.append(doc)
                probe_chars += len(doc)
                if len(probe_docs) >= 200:
                    break

            probe_encoded = encode_many(probe_docs, pool)
            probe_tokens = sum(len(x) for x in probe_encoded)
            tokens_per_char = probe_tokens / max(1, probe_chars)
            est_tokens = census[key]["est_chars"] * tokens_per_char

            # Oversample ~8%, then stop exactly when the requested token budget is reached.
            q_train = min(1.0, 1.08 * target / max(1.0, est_tokens))
            q_val = min(
                1.0,
                1.20 * VAL_TOKENS_PER_SOURCE /
                max(1.0, est_tokens * VAL_PERMILLE / 1000.0)
            )

            print(
                f"\n[{key}] est_tokens≈{est_tokens/1e6:.0f}M | "
                f"target={target/1e6:.0f}M | q_train={q_train:.5f} | q_val={q_val:.5f}",
                flush=True
            )

            n_train = 0
            n_val = 0
            docs_seen = 0
            gen_found = 0
            t0 = time.time()

            batch_texts = []
            batch_is_val = []

            def flush_batch():
                global n_train, n_val, batch_texts, batch_is_val
                if not batch_texts:
                    return

                arrays = encode_many(batch_texts, pool)

                for arr, is_val in zip(arrays, batch_is_val):
                    if is_val:
                        if n_val >= VAL_TOKENS_PER_SOURCE:
                            continue
                        remaining = VAL_TOKENS_PER_SOURCE - n_val
                        a = arr[:remaining]
                        a.tofile(f_val)
                        n_val += len(a)
                    else:
                        if n_train >= target:
                            continue
                        remaining = target - n_train
                        a = arr[:remaining]
                        a.tofile(f_train)
                        n_train += len(a)

                batch_texts = []
                batch_is_val = []

            for idx, doc in enumerate(iter_docs(path)):
                docs_seen += 1
                if len(doc) < 200:
                    continue

                # Deterministic document-level heldout split.
                is_val_doc = ((idx * 2654435761) % 1000) < VAL_PERMILLE

                if is_val_doc:
                    if n_val >= VAL_TOKENS_PER_SOURCE:
                        continue
                    if rng.random() > q_val:
                        continue

                    words = doc.split()
                    if gen_found < VAL_GEN_PER_SOURCE and len(words) >= 640:
                        st = rng.randint(0, len(words)-640)
                        exid = f"{key}_{gen_found:02d}"
                        val_gen[exid] = {"text":" ".join(words[st:st+512])}
                        val_ref[exid] = {"text":" ".join(words[st+512:st+640])}
                        gen_found += 1

                    batch_texts.append(doc)
                    batch_is_val.append(True)

                else:
                    if n_train >= target:
                        # Keep scanning only until validation is also sufficiently filled.
                        if n_val >= int(0.95 * VAL_TOKENS_PER_SOURCE):
                            break
                        continue
                    if rng.random() > q_train:
                        continue
                    batch_texts.append(doc)
                    batch_is_val.append(False)

                if len(batch_texts) >= 64:
                    flush_batch()

                    if n_train and n_train % 25_000_000 < 100_000:
                        print(
                            f"  [{key}] train={n_train/1e6:.1f}M "
                            f"val={n_val/1e6:.2f}M docs_seen={docs_seen:,} "
                            f"elapsed={(time.time()-t0)/60:.1f}m",
                            flush=True
                        )

            flush_batch()

            meta["sources"][key] = {
                "train_tokens":n_train,
                "val_tokens":n_val,
                "docs_seen":docs_seen,
                "tokens_per_char_probe":tokens_per_char,
                "q_train":q_train,
            }

            print(
                f"[{key}] DONE train={n_train/1e6:.1f}M "
                f"val={n_val/1e6:.2f}M local_gen={gen_found} "
                f"time={(time.time()-t0)/60:.1f}m"
            )

            maybe_delete_download(path)

    finally:
        if pool is not None:
            pool.close()
            pool.join()
        f_train.close()
        f_val.close()

    json.dump(val_gen, open(VAL_GEN_PATH,"w"), ensure_ascii=False, indent=2)
    json.dump(val_ref, open(VAL_REF_PATH,"w"), ensure_ascii=False, indent=2)

    meta["actual_train_tokens"] = os.path.getsize(TRAIN_BIN)//2
    meta["actual_val_tokens"] = os.path.getsize(VAL_BIN)//2
    json.dump(meta, open(META_PATH,"w"), indent=2)

    print("\nPreprocessing complete.")
    print("train tokens:", f"{meta['actual_train_tokens']/1e6:.1f}M")
    print("val tokens:", f"{meta['actual_val_tokens']/1e6:.2f}M")
    print("binary train size:", f"{os.path.getsize(TRAIN_BIN)/1e9:.2f} GB")
    print("local generation examples:", len(val_gen))


[sc] est_tokens≈474M | target=180M | q_train=0.41045 | q_val=0.25336
  [sc] train=100.1M val=0.30M docs_seen=29,723 elapsed=0.8m
  [sc] train=150.0M val=0.47M docs_seen=43,274 elapsed=1.2m
[sc] DONE train=180.0M val=0.50M local_gen=40 time=1.4m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/96d8e4d106292e7d5b6c4cb81e125f7e8cdee50b66302a373f547178b5364923

[hc] est_tokens≈3652M | target=180M | q_train=0.05323 | q_val=0.03286
  [hc] train=0.1M val=0.00M docs_seen=1,207 elapsed=0.0m
  [hc] train=50.0M val=0.17M docs_seen=735,322 elapsed=0.7m
  [hc] train=50.1M val=0.17M docs_seen=736,500 elapsed=0.7m
  [hc] train=100.0M val=0.40M docs_seen=1,211,507 elapsed=1.2m
  [hc] train=150.1M val=0.50M docs_seen=2,090,961 elapsed=1.9m
[hc] DONE train=180.0M val=0.50M local_gen=40 time=2.2m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/59b3d4df6b492dd03d89197aa634b58

In [8]:
# Cell 8 — Memory-mapped block dataset sanity
meta = json.load(open(META_PATH))
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_mm = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

BLOCK_TOKENS = SEQ_LEN + 1
N_TRAIN_BLOCKS = len(train_mm) // BLOCK_TOKENS
N_VAL_BLOCKS = len(val_mm) // BLOCK_TOKENS

print("Train token IDs:", f"{len(train_mm):,}")
print("Val token IDs:", f"{len(val_mm):,}")
print("Train blocks:", f"{N_TRAIN_BLOCKS:,}")
print("Val blocks:", f"{N_VAL_BLOCKS:,}")

assert N_TRAIN_BLOCKS > 0 and N_VAL_BLOCKS > 0
assert len(tokenizer) < 65536

sample = np.asarray(train_mm[:BLOCK_TOKENS], dtype=np.int64)
print("Sample block shape:", sample.shape)
print("Decoded start:", repr(tokenizer.decode(sample[:80])[:300]))

Train token IDs: 500,000,000
Val token IDs: 1,931,848
Train blocks: 325,309
Val blocks: 1,256
Sample block shape: (1537,)
Decoded start: 'JUDGMENT\n\nK.C. Das Gupta, J.\n\n1. The only question that arises for consideration in this appeal by the State Bank of India is as regards the effect of Para 292 of the Bank Award, as modified, on the respondent, '


In [9]:
# Cell 9 — Optimized ~120M Prefix Encoder LM with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class PrefixSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.transpose(1,2)
        k = k.transpose(1,2)
        v = v.transpose(1,2)

        if past_kv is not None:
            pk,pv = past_kv
            k = torch.cat([pk,k], dim=2)
            v = torch.cat([pv,v], dim=2)

        y = F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.out_proj(y)

        new_kv = (k,v) if use_cache else None
        return y, new_kv


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixSelfAttention(d_model,n_heads,dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model,ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim,d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a,new_kv = self.attn(
            self.ln1(x),
            attn_mask=attn_mask,
            past_kv=past_kv,
            use_cache=use_cache,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x,new_kv


class PrefixEncoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size,d_model)
        self.pos = SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        # Tied output projection: no second vocab-sized matrix.
        self.apply(self._init_weights)

        # GPT-style residual projection scaling improves deep-model stability.
        scaled_std = 0.02 / math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def prefix_mask(T, prefix_len, device):
        # True means attention is allowed for SDPA boolean mask.
        q = torch.arange(T, device=device)[:,None]
        k = torch.arange(T, device=device)[None,:]
        p = int(prefix_len)
        allowed = ((q < p) & (k < p)) | ((q >= p) & (k <= q))
        return allowed[None,None,:,:]

    def forward_hidden(self, input_ids, prefix_len=None, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        # position_ids / key_valid are only used for LEFT-PADDED BATCHED generation; when both are None the
        # behaviour is exactly the original (training path unchanged).
        B,T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len+T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len+T, device=input_ids.device
            )[None,:].expand(B,T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        # Full training sequence uses the prefix-LM mask.
        # Initial generation prefill has prefix_len == T, therefore it is fully bidirectional.
        # Cached one-token decoding needs no mask because only past/current KVs exist.
        if key_valid is not None:
            # (B,1,1,S) bool, True = real token. Prompt is bidirectional; padding is never attended to.
            attn_mask = key_valid[:,None,None,:]
        elif past_key_values is None and prefix_len is not None and int(prefix_len) < T:
            attn_mask = self.prefix_mask(T,prefix_len,input_ids.device)
        else:
            attn_mask = None

        new_cache = [] if use_cache else None

        for i,block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x,kv = block(
                x,
                attn_mask=attn_mask,
                past_kv=past,
                use_cache=use_cache,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x,new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)

model = PrefixEncoderLM(
    vocab_size=len(tokenizer),
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    ffn_dim=FFN_DIM,
    dropout=DROPOUT,
    max_positions=MAX_POSITIONS,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print("Target range requested: 100–150M")
assert 100_000_000 <= n_params <= 150_000_000, n_params

Model parameters: 118.11M
Target range requested: 100–150M


In [10]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt
/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt


In [11]:
from pathlib import Path
import shutil

src_latest = Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt")
src_best   = Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt")

dst_dir = Path("/kaggle/working/cs781_encoder_120m/checkpoints/enc_full")
dst_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(src_latest, dst_dir / "latest.pt")
shutil.copy2(src_best,   dst_dir / "best.pt")

print("Restored latest:", dst_dir / "latest.pt")
print("Restored best:", dst_dir / "best.pt")

Restored latest: /kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt
Restored best: /kaggle/working/cs781_encoder_120m/checkpoints/enc_full/best.pt


In [12]:
import torch

ckpt = torch.load(
    "/kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt",
    map_location="cpu",
    weights_only=False
)

print("Saved step:", ckpt.get("global_step", ckpt.get("step")))
print("Cursor:", ckpt.get("cursor"))
print("Best val:", ckpt.get("best_val"))

RuntimeError: PytorchStreamReader failed reading file .format_version: invalid header or archive is corrupted

In [13]:
from pathlib import Path
import os

paths = [
    Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt"),
    Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt"),
    Path("/kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt"),
]

for p in paths:
    print("\n", p)
    print("exists:", p.exists())
    if p.exists():
        print("size GB:", p.stat().st_size / (1024**3))


 /kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt
exists: True
size GB: 1.3201364381238818

 /kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt
exists: True
size GB: 0.4400355005636811

 /kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt
exists: True
size GB: 1.3201364381238818


In [14]:
import torch

for p in [
    "/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt",
    "/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt",
]:
    print("\nTesting:", p)
    try:
        obj = torch.load(p, map_location="cpu", weights_only=False)
        print("OK")
        print("keys:", obj.keys())
        print("step:", obj.get("global_step", obj.get("step")))
        print("cursor:", obj.get("cursor"))
    except Exception as e:
        print("FAILED:", repr(e))


Testing: /kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_latest (5).pt
FAILED: RuntimeError('PytorchStreamReader failed reading file .format_version: invalid header or archive is corrupted')

Testing: /kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt
OK
keys: dict_keys(['model', 'global_step', 'val_loss', 'config'])
step: 13000
cursor: None


In [15]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "enc_full_latest (4).pt":
            print(os.path.join(root, f))

/kaggle/input/datasets/aaryan1234567890/phase1-latest-asdsdd/enc_full_latest (4).pt


In [16]:
import torch

p = "/kaggle/input/datasets/aaryan1234567890/phase1-latest-asdsdd/enc_full_latest (4).pt"

try:
    obj = torch.load(p, map_location="cpu", weights_only=False)
    print("OK")
    print("keys:", obj.keys())
    print("step:", obj.get("global_step", obj.get("step")))
    print("cursor:", obj.get("cursor"))
    print("best_val:", obj.get("best_val"))
except Exception as e:
    print("FAILED:", repr(e))

OK
keys: dict_keys(['model', 'optimizer', 'scheduler', 'scaler', 'cursor', 'global_step', 'best_val', 'python_random_state', 'config'])
step: 13000
cursor: 208000
best_val: 2.8730723578482866


In [17]:
from pathlib import Path
import shutil

src = Path("/kaggle/input/datasets/aaryan1234567890/phase1-latest-asdsdd/enc_full_latest (4).pt")
dst = Path("/kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print("Restored latest checkpoint:", dst)

Restored latest checkpoint: /kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt


In [18]:
import torch

ckpt = torch.load(
    "/kaggle/working/cs781_encoder_120m/checkpoints/enc_full/latest.pt",
    map_location="cpu",
    weights_only=False
)

print("step:", ckpt["global_step"])
print("cursor:", ckpt["cursor"])

step: 13000
cursor: 208000


In [19]:
# Cell 10 — Full training loop: one shuffled pass over ~500M tokens, resumable
def get_block(mm, block_index):
    start = int(block_index) * BLOCK_TOKENS
    arr = np.asarray(mm[start:start+BLOCK_TOKENS], dtype=np.int64)
    return torch.from_numpy(arr)

def get_batch_from_indices(mm, indices):
    return torch.stack([get_block(mm,i) for i in indices], dim=0).to(DEVICE, non_blocking=True)

# Reserved id used as the MLM mask token. The byte-level tokenizer NEVER emits <s> (only <pad>/<unk>/<s> are specials, none
# is produced by encode()), and generation already bans it, so it is a free id: no tokenizer/data change is needed.
MASK_TOKEN_ID = tokenizer.bos_token_id

# Per-source validation boundaries inside val_all.bin (sources were written one after another in Cell 7).
_val_bounds, _cum = [], 0
for _k, _s in meta["sources"].items():
    _val_bounds.append((_k, _cum, _cum + int(_s["val_tokens"])))
    _cum += int(_s["val_tokens"])

def val_source_of_block(idx):
    start = int(idx) * BLOCK_TOKENS
    for k, lo, hi in _val_bounds:
        if lo <= start < hi:
            return k
    return "other"

LAST_VAL_DETAIL = {}

@torch.no_grad()
def evaluate_val(model, max_blocks=VAL_EVAL_BLOCKS):
    """Returns the overall prefix-LM validation loss (used for best-checkpoint selection, as before).
    Per-source prefix-LM losses and an MLM loss are stored in LAST_VAL_DETAIL for logging."""
    model.eval()
    n = min(N_VAL_BLOCKS, max_blocks)
    if n <= 0:
        return float("inf")

    # Evenly cover validation instead of only using the beginning.
    ids = np.linspace(0, N_VAL_BLOCKS-1, n, dtype=np.int64)

    losses = []
    by_src = {}
    for idx in ids:
        batch = get_batch_from_indices(val_mm,[idx])
        x = batch[:,:-1]
        y = batch[:,1:]

        # Evaluate at a real prompt-like absolute BPE prefix length.
        p = max(1, min(x.size(1), PROMPT_KEEP_TOKENS))

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
            h,_ = model.forward_hidden(x,prefix_len=p)
            start = max(0,p-1)
            # IMPORTANT SPEED/MEMORY OPTIMIZATION:
            # project only suffix positions that participate in the loss.
            logits = model.project(h[:,start:,:])
            targets = y[:,start:]
            loss = F.cross_entropy(
                logits.reshape(-1,logits.size(-1)),
                targets.reshape(-1),
            )
        lv = float(loss.item())
        losses.append(lv)
        by_src.setdefault(val_source_of_block(idx), []).append(lv)

    # MLM validation (fixed seed, <=64 blocks): health check of the bidirectional/encoder objective.
    mlm_losses = []
    if P_MLM > 0.0:
        g = torch.Generator(device=DEVICE)
        g.manual_seed(999)
        for idx in ids[::max(1, len(ids)//64)][:64]:
            x = get_batch_from_indices(val_mm,[idx])[:,:-1]
            m = (torch.rand(x.shape, device=x.device, generator=g) < MLM_PROB) & (x != tokenizer.eos_token_id)
            if not bool(m.any()):
                continue
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
                h,_ = model.forward_hidden(x.masked_fill(m, MASK_TOKEN_ID), prefix_len=None)
                l2 = F.cross_entropy(model.project(h[m]), x[m])
            mlm_losses.append(float(l2.item()))

    LAST_VAL_DETAIL.clear()
    LAST_VAL_DETAIL["val_lm_by_source"] = {k: sum(v)/len(v) for k,v in by_src.items()}
    LAST_VAL_DETAIL["val_mlm"] = (sum(mlm_losses)/len(mlm_losses)) if mlm_losses else None

    model.train()
    return sum(losses)/len(losses)


# Weight decay only on matrices (embeddings + Linear weights); NOT on biases / LayerNorm gains.
decay_params   = [p for n_, p in model.named_parameters() if p.requires_grad and p.ndim >= 2]
nodecay_params = [p for n_, p in model.named_parameters() if p.requires_grad and p.ndim < 2]
param_groups = [
    {"params": decay_params,   "weight_decay": WEIGHT_DECAY},
    {"params": nodecay_params, "weight_decay": 0.0},
]
# Fused AdamW is faster on recent CUDA PyTorch; fall back automatically.
optim_kwargs = dict(
    lr=LEARNING_RATE,
    betas=(0.9,0.95),
    eps=1e-8,
)
try:
    optimizer = torch.optim.AdamW(param_groups, fused=(DEVICE.type=="cuda"), **optim_kwargs)
    print("AdamW fused:", DEVICE.type=="cuda")
except Exception:
    optimizer = torch.optim.AdamW(param_groups, **optim_kwargs)
    print("Using standard AdamW.")

N_TRAIN_USED = N_TRAIN_BLOCKS if TRAIN_BLOCK_LIMIT is None else min(N_TRAIN_BLOCKS, int(TRAIN_BLOCK_LIMIT))
print(f"Training on {N_TRAIN_USED:,} of {N_TRAIN_BLOCKS:,} blocks (~{N_TRAIN_USED*SEQ_LEN/1e6:.0f}M tokens) | run={RUN_NAME}")
blocks_per_micro = MICRO_BATCH_SIZE
micro_steps_total = math.ceil(N_TRAIN_USED / MICRO_BATCH_SIZE)
opt_steps_total = math.ceil(micro_steps_total / GRAD_ACCUM_STEPS)
warmup_steps = max(1,int(opt_steps_total*WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-6, step/max(1,warmup_steps))
    if LR_SCHEDULE == "cosine":
        progress = (step-warmup_steps)/max(1,opt_steps_total-warmup_steps)
        cosine = 0.5*(1+math.cos(math.pi*min(1.0,progress)))
        return MIN_LR_RATIO + (1-MIN_LR_RATIO)*cosine
    # WSD: constant LR, then linear decay to MIN_LR_RATIO over the last WSD_DECAY_FRAC of the optimizer steps.
    decay_start = int(opt_steps_total*(1.0-WSD_DECAY_FRAC))
    if step < decay_start:
        return 1.0
    prog = (step-decay_start)/max(1,opt_steps_total-decay_start)
    return 1.0 + (MIN_LR_RATIO-1.0)*min(1.0,prog)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

# Deterministic block permutation: enables exact cursor-based resume without replaying DataLoader.
perm_gen = torch.Generator()
perm_gen.manual_seed(SEED + 777)
block_perm = torch.randperm(N_TRAIN_BLOCKS, generator=perm_gen).numpy()

cursor = 0
global_step = 0
best_val = float("inf")

def try_restore_from_hf():
    """If there is no local latest.pt (fresh Kaggle session) pull it from the private HF checkpoint repo."""
    if LATEST_CKPT.exists() or not HF_CHECKPOINT_REPO:
        return
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=f"{RUN_NAME}/latest.pt",
                            repo_type="model", token=HF_TOKEN)
        shutil.copy(p, LATEST_CKPT)
        try:
            os.remove(os.path.realpath(p))   # drop the cache copy (1.4 GB)
        except Exception:
            pass
        print("Restored latest.pt from HF checkpoint repo.")
    except Exception as e:
        print("No restorable checkpoint on HF (starting fresh):", repr(e)[:160])

try_restore_from_hf()

if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    cursor = int(ckpt["cursor"])
    global_step = int(ckpt["global_step"])
    best_val = float(ckpt["best_val"])
    if "python_random_state" in ckpt:
        random.setstate(ckpt["python_random_state"])
    print(f"RESUME: cursor={cursor:,}/{N_TRAIN_USED:,} | step={global_step:,} | best_val={best_val:.4f}")

def atomic_torch_save(obj, path):
    """Write completely to a temp file, then atomically replace the target."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def checkpoint_object():
    return {
        "model":model.state_dict(),
        "optimizer":optimizer.state_dict(),
        "scheduler":scheduler.state_dict(),
        "scaler":scaler.state_dict(),
        "cursor":cursor,
        "global_step":global_step,
        "best_val":best_val,
        "python_random_state":random.getstate(),
        "config":{
            "vocab_size":len(tokenizer),
            "d_model":D_MODEL,
            "n_layers":N_LAYERS,
            "n_heads":N_HEADS,
            "ffn_dim":FFN_DIM,
            "dropout":DROPOUT,
            "max_positions":MAX_POSITIONS,
            "seq_len":SEQ_LEN,
            "prompt_p10_tokens":PROMPT_P10_TOKENS,
            "prompt_p95_tokens":PROMPT_P95_TOKENS,
            "expected_cont_tokens":EXPECTED_CONT_TOKENS,
            "prefix_tokens_min":PREFIX_TOKENS_MIN,
            "prefix_tokens_max":PREFIX_TOKENS_MAX,
            "p_mlm":P_MLM,
            "mlm_prob":MLM_PROB,
            "mask_token_id":MASK_TOKEN_ID,
        },
    }

_hf_upload_count = 0

def maybe_upload_latest_to_hf():
    """Best-effort backup: NEVER lets a network/permission error kill training."""
    global _hf_upload_count
    if not HF_CHECKPOINT_REPO:
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_CHECKPOINT_REPO, repo_type="model", private=True, exist_ok=True)
        api.upload_file(
            path_or_fileobj=str(LATEST_CKPT),
            path_in_repo=f"{RUN_NAME}/latest.pt",
            repo_id=HF_CHECKPOINT_REPO,
            repo_type="model",
            commit_message=f"{RUN_NAME} checkpoint step {global_step}",
        )
        _hf_upload_count += 1
        print(f"Uploaded {RUN_NAME}/latest.pt to private HF checkpoint repo (upload #{_hf_upload_count}).")
        if HF_SQUASH_EVERY_N_UPLOADS and _hf_upload_count % HF_SQUASH_EVERY_N_UPLOADS == 0:
            try:
                api.super_squash_history(repo_id=HF_CHECKPOINT_REPO, repo_type="model")
                print("Squashed HF repo history (bounds storage).")
            except Exception as e:
                print("History squash skipped:", repr(e)[:120])
    except Exception as e:
        print("HF checkpoint upload FAILED (training continues):", repr(e)[:200])

LOG_JSONL = CKPT_DIR / "train_log.jsonl"
def append_log(rec):
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), **rec}
    with open(LOG_JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")

def save_checkpoint(path):
    atomic_torch_save(checkpoint_object(), path)

_CFG_NAMES = ["RUN_NAME","VOCAB_SIZE","SEQ_LEN","MAX_POSITIONS","D_MODEL","N_LAYERS","N_HEADS","FFN_DIM","DROPOUT",
              "PREFIX_TOKENS_MIN","PREFIX_TOKENS_MAX","PROMPT_KEEP_TOKENS","EXPECTED_CONT_TOKENS",
              "MICRO_BATCH_SIZE","GRAD_ACCUM_STEPS","LEARNING_RATE","MIN_LR_RATIO","WARMUP_RATIO","WEIGHT_DECAY",
              "GRAD_CLIP","LR_SCHEDULE","WSD_DECAY_FRAC","P_MLM","MLM_PROB","TRAIN_BLOCK_LIMIT","SEED"]
run_config = {k: globals()[k] for k in _CFG_NAMES}
run_config.update(n_params=n_params, n_train_blocks_used=int(N_TRAIN_USED),
                  opt_steps_total=int(opt_steps_total), resumed_at_step=int(global_step))
json.dump(run_config, open(CKPT_DIR/"run_config.json","w"), indent=2)   # keep for the report

model.train()
optimizer.zero_grad(set_to_none=True)

running_loss = 0.0
running_micro = 0
run_lm_sum = run_lm_n = run_mlm_sum = run_mlm_n = 0
train_start = time.time()
last_report_time = train_start
last_report_cursor = cursor

while cursor < N_TRAIN_USED:
    accum_loss = 0.0
    accum_count = 0

    for _ in range(GRAD_ACCUM_STEPS):
        if cursor >= N_TRAIN_USED:
            break

        inds = block_perm[cursor:min(cursor+MICRO_BATCH_SIZE, N_TRAIN_USED)]
        if len(inds) == 0:
            break

        batch = get_batch_from_indices(train_mm, inds)
        cursor += len(inds)

        x = batch[:,:-1]
        y = batch[:,1:]

        use_mlm = (P_MLM > 0.0) and (random.random() < P_MLM)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
            if use_mlm:
                # MLM micro-batch: fully bidirectional attention; MLM_PROB of the non-EOS tokens -> reserved mask id.
                m = (torch.rand(x.shape, device=x.device) < MLM_PROB) & (x != tokenizer.eos_token_id)
                if not bool(m.any()):
                    m[:, x.size(1)//2] = True
                h,_ = model.forward_hidden(x.masked_fill(m, MASK_TOKEN_ID), prefix_len=None)
                logits = model.project(h[m])          # project only the masked positions
                targets = x[m]
            else:
                # prefix-LM micro-batch (unchanged): bidirectional prefix, causal continuation, loss on continuation.
                p = random.randint(PREFIX_TOKENS_MIN, PREFIX_TOKENS_MAX)
                p = max(1, min(x.size(1), p))
                start = max(0,p-1)
                h,_ = model.forward_hidden(x,prefix_len=p)
                logits = model.project(h[:,start:,:])
                targets = y[:,start:]
            loss = F.cross_entropy(
                logits.reshape(-1,logits.size(-1)),
                targets.reshape(-1),
            )
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        lv = float(loss.item())
        accum_loss += lv
        accum_count += 1
        if use_mlm:
            run_mlm_sum += lv; run_mlm_n += 1
        else:
            run_lm_sum += lv; run_lm_n += 1

    if accum_count == 0:
        break

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    global_step += 1
    running_loss += accum_loss/accum_count
    running_micro += 1

    if global_step % LOG_EVERY == 0:
        now = time.time()
        blocks_done = cursor - last_report_cursor
        tokens_done = blocks_done * SEQ_LEN
        tok_s = tokens_done / max(1e-6, now-last_report_time)

        remaining_tokens = max(0,N_TRAIN_USED-cursor)*SEQ_LEN
        eta_h = remaining_tokens/max(tok_s,1e-9)/3600

        avg = running_loss/max(1,running_micro)
        lm_avg = run_lm_sum/max(1,run_lm_n)
        mlm_avg = run_mlm_sum/max(1,run_mlm_n)
        lr = optimizer.param_groups[0]["lr"]

        print(
            f"step={global_step:,}/{opt_steps_total:,} | "
            f"blocks={cursor:,}/{N_TRAIN_USED:,} | "
            f"processed≈{cursor*SEQ_LEN/1e6:.1f}M tok | "
            f"loss={avg:.4f} | lm={lm_avg:.4f} mlm={mlm_avg:.4f} | lr={lr:.2e} | "
            f"{tok_s:,.0f} tok/s | ETA≈{eta_h:.1f}h",
            flush=True
        )

        append_log({"type":"train","step":global_step,"tokens":int(cursor*SEQ_LEN),"loss":avg,
                    "lm_loss":lm_avg if run_lm_n else None,"mlm_loss":mlm_avg if run_mlm_n else None,
                    "lr":lr,"tok_s":tok_s})
        running_loss = 0.0
        running_micro = 0
        run_lm_sum = run_lm_n = run_mlm_sum = run_mlm_n = 0
        last_report_time = now
        last_report_cursor = cursor

    if global_step % VAL_EVERY == 0 or cursor >= N_TRAIN_USED:
        val_loss = evaluate_val(model)
        ppl = math.exp(min(20,val_loss))
        print(f"VALIDATION step={global_step:,}: loss={val_loss:.4f} ppl={ppl:.2f}")
        print("  per-source prefix-LM val loss:",
              {k: round(v,4) for k,v in LAST_VAL_DETAIL.get("val_lm_by_source",{}).items()},
              "| MLM val loss:", None if LAST_VAL_DETAIL.get("val_mlm") is None else round(LAST_VAL_DETAIL["val_mlm"],4))
        append_log({"type":"val","step":global_step,"tokens":int(cursor*SEQ_LEN),"val_lm":val_loss,"val_ppl":ppl,
                    **LAST_VAL_DETAIL})

        if val_loss < best_val:
            best_val = val_loss
            atomic_torch_save({
                "model":model.state_dict(),
                "global_step":global_step,
                "val_loss":best_val,
                "config":{
                    "vocab_size":len(tokenizer),
                    "d_model":D_MODEL,
                    "n_layers":N_LAYERS,
                    "n_heads":N_HEADS,
                    "ffn_dim":FFN_DIM,
                    "dropout":DROPOUT,
                    "max_positions":MAX_POSITIONS,
                    "seq_len":SEQ_LEN,
                    "prompt_p10_tokens":PROMPT_P10_TOKENS,
                    "prompt_p95_tokens":PROMPT_P95_TOKENS,
                    "expected_cont_tokens":EXPECTED_CONT_TOKENS,
                },
            }, BEST_CKPT)
            print("Saved new BEST checkpoint.")
            # Also expose BEST checkpoint at Kaggle working root
            best_outside = Path("/kaggle/working/enc_full_best.pt")
            shutil.copy2(BEST_CKPT, best_outside)
            print("Updated external BEST checkpoint copy:", best_outside)

    if (global_step % SAVE_EVERY == 0 or cursor >= N_TRAIN_USED
            or (HF_CHECKPOINT_REPO and global_step % HF_UPLOAD_EVERY == 0)):
        save_checkpoint(LATEST_CKPT)
        print("Saved resumable latest checkpoint atomically.")
                # Also expose latest checkpoint at Kaggle working root
        import shutil
        
        outside_ckpt = Path("/kaggle/working/enc_full_latest.pt")
        shutil.copy2(LATEST_CKPT, outside_ckpt)
        
        print("Updated external checkpoint copy:", outside_ckpt)

    if HF_CHECKPOINT_REPO and (global_step % HF_UPLOAD_EVERY == 0 or cursor >= N_TRAIN_USED):
        maybe_upload_latest_to_hf()

save_checkpoint(FINAL_CKPT)
print("Training complete.")
print("Final checkpoint:", FINAL_CKPT)
print("Best validation loss:", best_val)
print("Total elapsed hours:", (time.time()-train_start)/3600)

AdamW fused: True
Training on 325,309 of 325,309 blocks (~500M tokens) | run=enc_full
RESUME: cursor=208,000/325,309 | step=13,000 | best_val=2.8731
step=13,050/20,332 | blocks=208,800/325,309 | processed≈320.7M tok | loss=2.8703 | lm=2.7147 mlm=3.1593 | lr=3.00e-04 | 13,165 tok/s | ETA≈3.8h
step=13,100/20,332 | blocks=209,600/325,309 | processed≈321.9M tok | loss=2.8626 | lm=2.7356 mlm=3.1589 | lr=3.00e-04 | 13,323 tok/s | ETA≈3.7h
step=13,150/20,332 | blocks=210,400/325,309 | processed≈323.2M tok | loss=2.8872 | lm=2.7737 mlm=3.1076 | lr=3.00e-04 | 13,444 tok/s | ETA≈3.6h
step=13,200/20,332 | blocks=211,200/325,309 | processed≈324.4M tok | loss=2.8712 | lm=2.7517 mlm=3.1311 | lr=3.00e-04 | 13,385 tok/s | ETA≈3.6h
step=13,250/20,332 | blocks=212,000/325,309 | processed≈325.6M tok | loss=2.8593 | lm=2.7584 mlm=3.1700 | lr=3.00e-04 | 13,244 tok/s | ETA≈3.7h
step=13,300/20,332 | blocks=212,800/325,309 | processed≈326.9M tok | loss=2.8857 | lm=2.7458 mlm=3.2198 | lr=3.00e-04 | 13,295 tok/

In [20]:
# Cell 11 — Reload best model + BATCHED KV-cache generation (left-padded; exactly 128 words when achievable)
best_obj = torch.load(BEST_CKPT if BEST_CKPT.exists() else FINAL_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_obj["model"])
model.eval()

def mask_specials(logits, block_eos=True):
    logits = logits.clone()
    banned = [
        tokenizer.pad_token_id,
        tokenizer.unk_token_id,
        tokenizer.bos_token_id,
    ]
    if block_eos:
        banned.append(tokenizer.eos_token_id)
    logits[:,banned] = -float("inf")
    return logits

def apply_rep_penalty(logits, generated, penalty):
    """generated: LongTensor (B, t) of tokens generated so far. Prompt tokens are NOT penalised."""
    if penalty == 1.0 or generated is None or generated.size(1) == 0:
        return logits
    sc = logits.gather(1, generated)
    return logits.scatter(1, generated, torch.where(sc < 0, sc*penalty, sc/penalty))

def sample_next(logits):
    if GEN_TEMPERATURE <= 0:
        return logits.argmax(-1,keepdim=True)

    logits = logits / GEN_TEMPERATURE

    if GEN_TOP_K > 0:
        k=min(GEN_TOP_K,logits.size(-1))
        kth=torch.topk(logits,k,dim=-1).values[:,-1:]
        logits=logits.masked_fill(logits<kth,float("-inf"))

    if GEN_TOP_P < 1.0:
        sl,si=torch.sort(logits,descending=True,dim=-1)
        pr=torch.softmax(sl,dim=-1)
        remove=(pr.cumsum(-1)-pr)>GEN_TOP_P
        sl=sl.masked_fill(remove,float("-inf"))
        logits=torch.full_like(logits,float("-inf"))
        logits.scatter_(1,si,sl)

    return torch.multinomial(torch.softmax(logits,-1),1)

GEN_COUNTERS = {"total":0, "short":0, "fallback":0}

@torch.inference_mode()
def generate_batch(prompts, target_words=TARGET_GENERATION_WORDS):
    """prompts: list[str] -> list[str]. Prompts are LEFT-padded into one batch (per-sample positions + key padding
    mask, so results equal one-at-a-time generation). Every sample stops at `target_words` words.
    Normal case: generation stays inside the trained positions (< SEQ_LEN). Only if a sample still has fewer than
    target_words words does it keep going up to MAX_POSITIONS (fallback, counted in GEN_COUNTERS). Never raises."""
    B = len(prompts)
    enc = [tokenizer.encode(p)[-PROMPT_KEEP_TOKENS:] for p in prompts]
    lens = [len(e) for e in enc]
    S_pad = max(lens)

    ids = torch.full((B, S_pad), tokenizer.pad_token_id, dtype=torch.long)
    valid = torch.zeros(B, S_pad, dtype=torch.bool)
    for b, e in enumerate(enc):
        ids[b, S_pad-len(e):] = torch.tensor(e, dtype=torch.long)
        valid[b, S_pad-len(e):] = True
    ids, valid = ids.to(DEVICE), valid.to(DEVICE)

    pos = (valid.long().cumsum(1) - 1).clamp(min=0)       # every real prompt starts at position 0, as in training
    next_pos = valid.sum(1)                                # (B,) position of the next token to generate
    hard_cap = max(1, min(GEN_MAX_NEW_TOKENS, MAX_POSITIONS - S_pad))

    amp = torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda"))
    with amp:
        h, cache = model.forward_hidden(ids, prefix_len=None, use_cache=True, position_ids=pos, key_valid=valid)
    last_h = h[:, -1, :]

    gen = torch.empty(B, 0, dtype=torch.long, device=DEVICE)
    done = [False]*B
    done_step = [None]*B
    texts = [""]*B

    for step in range(hard_cap):
        logits = model.project(last_h).float()
        logits = mask_specials(logits, block_eos=True)
        logits = apply_rep_penalty(logits, gen, GEN_REP_PENALTY)
        nt = sample_next(logits)                                           # (B,1)
        gen = torch.cat([gen, nt], dim=1)
        valid = torch.cat([valid, torch.ones(B, 1, dtype=torch.bool, device=DEVICE)], dim=1)

        if (step+1) % 8 == 0 or step+1 == hard_cap:
            for b in range(B):
                if not done[b]:
                    words = tokenizer.decode(gen[b].tolist()).split()
                    if len(words) >= target_words:
                        done[b] = True
                        done_step[b] = step+1
                        texts[b] = " ".join(words[:target_words])
            if all(done):
                break

        with amp:
            h, cache = model.forward_hidden(nt, past_key_values=cache, use_cache=True,
                                            position_ids=next_pos[:, None], key_valid=valid)
        last_h = h[:, -1, :]
        next_pos = next_pos + 1

    for b in range(B):
        GEN_COUNTERS["total"] += 1
        if not done[b]:
            words = tokenizer.decode(gen[b].tolist()).split()
            texts[b] = " ".join(words[:target_words])
            GEN_COUNTERS["short"] += 1
        elif lens[b] + done_step[b] > SEQ_LEN:
            GEN_COUNTERS["fallback"] += 1
    return texts

def generate_continuation(prompt, target_words=TARGET_GENERATION_WORDS):
    return generate_batch([prompt], target_words)[0]

def generate_many(texts, desc="gen"):
    """texts: list[str] -> list[str] (same order). Sorted by token length so left-padding stays small."""
    order = sorted(range(len(texts)), key=lambda i: len(tokenizer.encode(texts[i])))
    out = [None]*len(texts)
    t0 = time.time()
    for s in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[s:s+GEN_BATCH_SIZE]
        for i, r in zip(idx, generate_batch([texts[i] for i in idx])):
            out[i] = r
        print(f"  {desc}: {min(s+GEN_BATCH_SIZE, len(order))}/{len(order)} | {time.time()-t0:.0f}s", flush=True)
    return out

val_gen = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
GEN_COUNTERS.update(total=0, short=0, fallback=0)

t0 = time.time()
val_ids = list(val_gen.keys())
val_outs = generate_many([val_gen[k]["text"] for k in val_ids], desc="val")
local_predictions = {k: {"text": o} for k, o in zip(val_ids, val_outs)}

for k in val_ids[:3]:
    print(f"{k} | words={len(local_predictions[k]['text'].split())} | {local_predictions[k]['text'][:160]!r}")
print("generation counters:", GEN_COUNTERS,
      "(short = < target words even after fallback; fallback = needed positions beyond SEQ_LEN)")

json.dump(local_predictions, open(LOCAL_PRED, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Generation elapsed:", f"{time.time()-t0:.1f}s")
print("Saved:", LOCAL_PRED)


  val: 16/155 | 8s
  val: 32/155 | 16s
  val: 48/155 | 24s
  val: 64/155 | 31s
  val: 80/155 | 39s
  val: 96/155 | 47s
  val: 112/155 | 55s
  val: 128/155 | 63s
  val: 144/155 | 73s
  val: 155/155 | 80s
sc_00 | words=128 | 'and the respondent was a respondent, and that it was not at all three or four payment of the disputed income of the capital tax and there was no impediment in t'
sc_01 | words=128 | 'court has held that the matter has not been taken into consideration the evidence by the respondents. Section 14 of the Act provides for the purpose of applicat'
sc_02 | words=128 | 'by recording of the allegations of the witnesses and the accused would be referred to an applicant in a criminal case and with the investigation of the case wit'
generation counters: {'total': 155, 'short': 0, 'fallback': 0} (short = < target words even after fallback; fallback = needed positions beyond SEQ_LEN)
Generation elapsed: 80.6s
Saved: /kaggle/working/cs781_encoder_120m/val_pred.json


In [21]:
# Cell 12 — Local Phase-1 evaluator: BERTScore + ROUGE-L + BLEU-4
def lcs_length(a,b):
    if not a or not b: return 0
    if len(a)<len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a):
        masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0)
        x=m|state
        y=(state<<1)|1
        state=x & ~(x-y)
    return state.bit_count()

def rouge_l_f1(pred,ref):
    pt,rt=pred.split(),ref.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    p,r=l/len(pt),l/len(rt)
    return 2*p*r/(p+r)

def ngrams(t,n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds,refs):
    clipped=[0]*4
    totals=[0]*4
    pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,5):
            pn,rn=ngrams(p,n),ngrams(r,n)
            totals[n-1]+=sum(pn.values())
            for g,c in pn.items():
                clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/4)

truth=json.load(open(VAL_REF_PATH,encoding="utf-8"))
pred=json.load(open(LOCAL_PRED,encoding="utf-8"))

if set(truth)!=set(pred):
    raise ValueError(f"ID mismatch: missing={set(truth)-set(pred)}, extra={set(pred)-set(truth)}")

ids=list(truth)
P=[pred[i]["text"].strip() for i in ids]
R=[truth[i]["text"].strip() for i in ids]

for i,p in zip(ids,P):
    if not p:
        raise ValueError(f"empty prediction for {i}")

rl=sum(rouge_l_f1(p,r) for p,r in zip(P,R))/len(P)
b4=bleu4([p.split() for p in P],[r.split() for r in R])
out={"ROUGE-L F1":rl,"BLEU-4":b4}

bert_per_sample=None
if not FAST_EVAL_NO_BERT:
    import bert_score
    _,_,f=bert_score.score(
        P,R,
        lang="en",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=False,
        device=str(DEVICE),
    )
    bert_per_sample=f.tolist()
    out={"BERTScore F1":float(f.mean().item()),**out}

print("examples:",len(ids))
print("mean words/pred:",sum(len(p.split()) for p in P)/len(P))
print(json.dumps({k:round(v,6) for k,v in out.items()},indent=2))
# ---- per-source breakdown (ids look like "sc_00", "hc_03", ...) -> needed for the data-mixing analysis
def _src(i): return i.split("_")[0]
print("\nPer-source results:")
for s_ in sorted(set(_src(i) for i in ids)):
    ix = [k for k,i in enumerate(ids) if _src(i)==s_]
    row = {"n":len(ix), "ROUGE-L F1": round(sum(rouge_l_f1(P[k],R[k]) for k in ix)/len(ix), 4)}
    if bert_per_sample is not None:
        row["BERTScore F1"] = round(sum(bert_per_sample[k] for k in ix)/len(ix), 4)
    print(" ", s_, row)

# ---- append this decoding setting + scores to a log (use it for the decoding sweep table in the report)
with open(ROOT/"eval_log.jsonl","a") as _f:
    _f.write(json.dumps({"time":time.strftime("%Y-%m-%d %H:%M:%S"), "run":RUN_NAME,
                         "temperature":GEN_TEMPERATURE, "top_p":GEN_TOP_P, "top_k":GEN_TOP_K,
                         "rep_penalty":GEN_REP_PENALTY, "n":len(ids),
                         **{k:float(v) for k,v in out.items()}}) + "\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


examples: 155
mean words/pred: 128.0
{
  "BERTScore F1": 0.819721,
  "ROUGE-L F1": 0.187046,
  "BLEU-4": 0.029011
}

Per-source results:
  acts {'n': 35, 'ROUGE-L F1': 0.1973, 'BERTScore F1': 0.8085}
  hc {'n': 40, 'ROUGE-L F1': 0.1893, 'BERTScore F1': 0.8234}
  sc {'n': 40, 'ROUGE-L F1': 0.1959, 'BERTScore F1': 0.823}
  tr {'n': 40, 'ROUGE-L F1': 0.167, 'BERTScore F1': 0.8226}


In [24]:
RUN_OFFICIAL_TEST = True

In [25]:
# Cell 13 — OPTIONAL official test generation + Codabench submission.zip
# Batched, incremental and resumable: a partial file is written during generation, a short continuation never aborts
# the run (it is reported instead), and re-running continues where it stopped.
if RUN_OFFICIAL_TEST:
    test_path=find_test_file()
    test=json.load(open(test_path,encoding="utf-8"))
    test={str(k):v for k,v in test.items()}

    # A partial file is only reused if it came from the SAME checkpoint + decoding settings.
    fingerprint = {
        "ckpt_step": best_obj.get("global_step"),
        "temperature": GEN_TEMPERATURE, "top_p": GEN_TOP_P, "top_k": GEN_TOP_K,
        "rep_penalty": GEN_REP_PENALTY, "prompt_keep": PROMPT_KEEP_TOKENS, "seq_len": SEQ_LEN,
    }
    partial_path = ROOT / "prediction_partial.json"

    def atomic_json(obj, path):
        tmp = Path(str(path) + ".tmp")
        json.dump(obj, open(tmp,"w",encoding="utf-8"), ensure_ascii=False)
        os.replace(tmp, path)

    predictions = {}
    if partial_path.exists():
        saved = json.load(open(partial_path, encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint:
            predictions = saved["predictions"]
            print(f"Resuming official generation: {len(predictions)} prompts already done.")
        else:
            print("Ignoring stale partial file (different checkpoint/decoding settings).")

    todo = [k for k in test if k not in predictions]
    todo.sort(key=lambda k: len(tokenizer.encode(test[k]["text"])))
    print(f"Official prompts: {len(test)} | to generate now: {len(todo)}")

    GEN_COUNTERS.update(total=0, short=0, fallback=0)
    t0=time.time()
    for n, s in enumerate(range(0, len(todo), GEN_BATCH_SIZE), 1):
        chunk = todo[s:s+GEN_BATCH_SIZE]
        outs = generate_batch([test[k]["text"] for k in chunk])
        for k, o in zip(chunk, outs):
            predictions[k] = {"text": o}
        if n % OFFICIAL_PARTIAL_EVERY == 0 or s+GEN_BATCH_SIZE >= len(todo):
            atomic_json({"fingerprint":fingerprint, "predictions":predictions}, partial_path)
        done_n = len(predictions)
        print(f"{done_n}/{len(test)} | elapsed={time.time()-t0:.0f}s", flush=True)

    if set(predictions)!=set(test.keys()):
        raise RuntimeError("official test ID mismatch")

    wc_bad = {k: len(v["text"].split()) for k, v in predictions.items() if len(v["text"].split()) != TARGET_GENERATION_WORDS}
    empty = [k for k, n in wc_bad.items() if n == 0]
    print("generation counters:", GEN_COUNTERS)
    if wc_bad:
        print(f"WARNING: {len(wc_bad)} prompts do not have exactly {TARGET_GENERATION_WORDS} words (not aborting): "
              f"{dict(list(wc_bad.items())[:10])}")
    if empty:
        raise RuntimeError(f"{len(empty)} empty predictions (partial results are saved): {empty[:5]}")

    json.dump(predictions,open(OFFICIAL_PRED,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

    with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(OFFICIAL_PRED,arcname="prediction.json")

    print("prediction.json:",OFFICIAL_PRED)
    print("submission.zip:",SUBMISSION_ZIP)
else:
    print("RUN_OFFICIAL_TEST=False — official test generation intentionally skipped.")


Official test file: Phase_1_Pretraining/Test/test.json
Official prompts: 1000 | to generate now: 1000
16/1000 | elapsed=7s
32/1000 | elapsed=15s
48/1000 | elapsed=23s
64/1000 | elapsed=31s
80/1000 | elapsed=38s
96/1000 | elapsed=45s
112/1000 | elapsed=52s
128/1000 | elapsed=60s
144/1000 | elapsed=68s
160/1000 | elapsed=75s
176/1000 | elapsed=83s
192/1000 | elapsed=90s
208/1000 | elapsed=98s
224/1000 | elapsed=106s
240/1000 | elapsed=114s
256/1000 | elapsed=122s
272/1000 | elapsed=129s
288/1000 | elapsed=137s
304/1000 | elapsed=144s
320/1000 | elapsed=152s
336/1000 | elapsed=160s
352/1000 | elapsed=168s
368/1000 | elapsed=176s
384/1000 | elapsed=183s
400/1000 | elapsed=191s
416/1000 | elapsed=198s
432/1000 | elapsed=206s
448/1000 | elapsed=214s
464/1000 | elapsed=222s
480/1000 | elapsed=230s
496/1000 | elapsed=237s
512/1000 | elapsed=245s
528/1000 | elapsed=253s
544/1000 | elapsed=260s
560/1000 | elapsed=268s
576/1000 | elapsed=275s
592/1000 | elapsed=283s
608/1000 | elapsed=290s
624/10

In [26]:
import zipfile, json

zip_path = "/kaggle/working/cs781_encoder_120m/submission.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:", z.namelist())
    data = json.loads(z.read("prediction.json"))

print("Number of predictions:", len(data))

first_id = next(iter(data))
print("First ID:", first_id)
print("First record:", data[first_id])
print("Word count:", len(data[first_id]["text"].split()))

ZIP contents: ['prediction.json']
Number of predictions: 1000
First ID: 775
First record: {'text': 'Section 28 of the Act. No doubt, the aforesaid persons who have been impleaded in the said suits filed by the High Court during the pendency of the said application. Therefore, in the absence of the main facts and circumstances of the case, we find no reason to interfere with the judgment of the High Court. In this case, we hold that the High Court was right in holding that the High Court was not right in rejecting the matter on the ground of the Trial Court and that the High Court erred in holding that the High Court had been justified in holding that the High Court had taken the decision of the High Court and the High Court and that in view of the aforesaid facts'}
Word count: 128


In [27]:
import torch
import os

files = [
    "/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_latest (11).pt",
    "/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_best (9).pt",
]

for p in files:
    print("\nTesting:", p)
    try:
        obj = torch.load(p, map_location="cpu", weights_only=False)
        print("✅ VALID")
        print("step:", obj.get("global_step", obj.get("step")))
        print("cursor:", obj.get("cursor"))
        print("best_val:", obj.get("best_val"))
        print("val_loss:", obj.get("val_loss"))
        print("size GB:", round(os.path.getsize(p)/(1024**3), 3))
    except Exception as e:
        print("❌ FAILED:", repr(e))


Testing: /kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_latest (11).pt
✅ VALID
step: 20332
cursor: 325309
best_val: 2.3875660663470626
val_loss: None
size GB: 1.32

Testing: /kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_best (9).pt
✅ VALID
step: 20332
cursor: None
best_val: None
val_loss: 2.3875660663470626
size GB: 0.44


In [28]:
import torch

files = {
    "latest": "/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_latest (11).pt",
    "best": "/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_best (9).pt",
}

for name, path in files.items():
    print("\n" + "="*80)
    print(name.upper(), path)

    obj = torch.load(path, map_location="cpu", weights_only=False)

    print("Top-level keys:")
    print(list(obj.keys()))

    for k, v in obj.items():
        print(f"\n{k}:")
        print("  type:", type(v))

        if isinstance(v, dict):
            print("  nested keys:", list(v.keys())[:30])
            if len(v) > 30:
                print("  ... total nested keys:", len(v))

        elif hasattr(v, "shape"):
            print("  shape:", v.shape)

        else:
            print("  value:", v)


LATEST /kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_latest (11).pt
Top-level keys:
['model', 'optimizer', 'scheduler', 'scaler', 'cursor', 'global_step', 'best_val', 'python_random_state', 'config']

model:
  type: <class 'collections.OrderedDict'>
  nested keys: ['token_emb.weight', 'blocks.0.ln1.weight', 'blocks.0.ln1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.out_proj.weight', 'blocks.0.attn.out_proj.bias', 'blocks.0.ln2.weight', 'blocks.0.ln2.bias', 'blocks.0.ff.0.weight', 'blocks.0.ff.0.bias', 'blocks.0.ff.2.weight', 'blocks.0.ff.2.bias', 'blocks.1.ln1.weight', 'blocks.1.ln1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.out_proj.weight', 'blocks.1.attn.out_proj.bias', 'blocks.1.ln2.weight', 'blocks.1.ln2.bias', 'blocks.1.ff.0.weight', 'blocks.1.ff.0.bias', 'blocks.1.ff.2.weight', 'blocks.1.ff.2.bias', 'blocks.2.ln1.weight', 'blocks.2.ln1.bias', 'blocks.2.attn.qkv.weight', 'blocks.2.attn.qkv.b

## Final checks before treating this as Architecture 1

You should see:

- tokenizer round-trip exact = `True`;
- vocabulary close to 24,576;
- Cell 6 prints a sensible `CHOSEN SEQ_LEN` and a prefix range that spans your real prompt lengths;
- all four sources represented in `meta.json`;
- actual training token count close to 500M;
- model parameter count between 100M and 150M;
- training: `lm=` and `mlm=` both decreasing; validation loss/perplexity improving (per source + MLM printed);
- local generated continuations reaching 128 words (`generation counters: short` should be ~0);
- BERTScore / ROUGE-L / BLEU completing (plus the per-source table);
- checkpoints written regularly, and (if enabled) uploaded to the private HF repo.

### Recommended order
1. **CPU session** (Accelerator = None, costs no GPU quota): Cells 1-8. Paste Cell 6's printout + the preprocessing summary.
2. **GPU pilot**: set `RUN_NAME = "enc_pilot"` and `TRAIN_BLOCK_LIMIT = 30_000_000 // SEQ_LEN`, run Cells 9-12. Check that loss falls,
   generations are readable and `short` is ~0. Look at the tokens/s line to estimate the full-run time.
3. **Full run**: `RUN_NAME = "enc_full"`, `TRAIN_BLOCK_LIMIT = None`, set `HF_CHECKPOINT_REPO` to a private repo you own.
   If the session dies, just re-run Cells 1-10: the checkpoint is restored from the repo and training resumes at the same data cursor.

### Practical notes
The **data/tokenizer stage is CPU-heavy**, while model training is GPU-heavy. Save `/kaggle/working/cs781_encoder_120m/data` as a Kaggle
dataset (or upload it to a private HF dataset repo) and attach it to the GPU notebook so Cells 1-8 need not be repeated.

Training uses one GPU. With the WSD schedule you can stop early (the LR will already be decaying only in the final 20%),
or extend a run later by raising the budget and resuming.
